# ETAPA 2: Análisis exploratorio y perfilado
# Objetivo: Explorar la evolución de los datos y realizar análisis comparativos.
# Rango temporal: 6 meses de datos (enero–junio 2020).
## Dificultad: Media – combinar múltiples archivos y realizar cálculos agregados. bold text

In [8]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
from IPython.display import display

In [2]:
# Define the start and end dates (6 primeros meses del 2021)
start_date = '2021-01-01'
end_date = '2021-6-30'

date_range = pd.date_range(start=start_date, end=end_date)

# print(date_range)

df = []

In [3]:
# Importar 6 primeros meses del 2021

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df.append(pd.read_csv(url))

In [4]:
# Concatenación
seisMeses = pd.concat(df, ignore_index=True)

In [ ]:
print("antes de optimizar")
seisMeses.info(memory_usage='deep')

In [ ]:
try:
    # Columnas de texto repetitivo -> 'category' (mayor ahorro de memoria)
    seisMeses['Country_Region'] = seisMeses['Country_Region'].astype('category')
    seisMeses['Province_State'] = seisMeses['Province_State'].astype('category')
    seisMeses['Admin2'] = seisMeses['Admin2'].astype('category')

    # Columnas numéricas (Rellenamos NaN con 0 y convertimos a entero de 32 bits)
    seisMeses['Confirmed'] = pd.to_numeric(seisMeses['Confirmed'], errors='coerce').fillna(0).astype('int32')
    seisMeses['Deaths'] = pd.to_numeric(seisMeses['Deaths'], errors='coerce').fillna(0).astype('int32')

    # Columnas con decimales o que pueden tener NaN (Convertimos a float de 32 bits)
    seisMeses['Recovered'] = pd.to_numeric(seisMeses['Recovered'], errors='coerce').astype('float32')
    seisMeses['Active'] = pd.to_numeric(seisMeses['Active'], errors='coerce').astype('float32')
    seisMeses['Lat'] = pd.to_numeric(seisMeses['Lat'], errors='coerce').astype('float32')
    seisMeses['Long_'] = pd.to_numeric(seisMeses['Long_'], errors='coerce').astype('float32')

    # Fechas
    seisMeses['Last_Update'] = pd.to_datetime(seisMeses['Last_Update'], errors='coerce')
    
    print("\n¡Optimización completa!")

except KeyError as e:
    print(f"Error: La columna {e} no se encontró. Es posible que el DataFrame esté vacío o los nombres de columna hayan cambiado.")
except Exception as e:
    print(f"Ocurrió un error inesperado durante la conversión: {e}")


In [ ]:
print("despues de optimizar")
seisMeses.info(memory_usage='deep')

1. ¿Cuáles son los 10 países con más casos confirmados acumulados durante el semestre?

In [ ]:
totalPaisConfirmados = seisMeses.groupby('Country_Region')['Confirmed'].sum().sort_values(ascending=False)
top10 = totalPaisConfirmados.head(10)
print('Top 10 países con más casos confirmados acumulados al 2020-06-30:')
display(top10.reset_index().rename(columns={'6/30/20':'Confirmado acumulado'}))

2. ¿Qué países presentan mayor tasa de letalidad (Deaths / Confirmed * 100)?

In [ ]:
%%time

tabla_letalidad = seisMeses.groupby('Country_Region')[['Deaths', 'Confirmed']].sum()
tabla_letalidad = tabla_letalidad[tabla_letalidad['Confirmed'] > 0].copy()  # evitar división por cero
tabla_letalidad['tasa_letalidad'] = (tabla_letalidad['Deaths'] / tabla_letalidad['Confirmed']) * 100
tabla_letalidad = tabla_letalidad.rename(columns={'Confirmed': "Confirmados", 'Deaths': "Muertes"})
tabla_letalidad.sort_values(by='tasa_letalidad', ascending=False).head(10)

3. ¿Cuántos países no registran recuperados en los datos analizados?

In [ ]:
%%time
#Revisar bien porque cuenta todos los 0, no solo los del recuperado, es una serie
totalRecuperadosPais = seisMeses.groupby('Country_Region')['Recovered'].sum()
NoRecuperados = (totalRecuperadosPais == 0).sum()
print('En total hay ',NoRecuperados,' paises que no registran recuperados en esos 6 meses')

4. ¿Qué país latinoamericano presenta la mayor cantidad de casos activos en junio 2020?

In [ ]:
%%time
tablaActivos = pd.merge(tabla_letalidad,totalRecuperadosPais, on='Country_Region',how='outer')
tablaActivos = tablaActivos.rename(columns={'Recovered': "Recuperados"})
tablaActivos['Activos'] = (tablaActivos['Confirmados'] - tablaActivos['Muertes'] - tablaActivos['Recuperados']).clip(lower=0)
mayorActivos = tablaActivos['Activos'].idxmax()
print('El pais latinoamericano con la mayor cantidad de casos activos en junio del 2020 es',mayorActivos)

5. ¿Cómo evolucionaron los casos confirmados en Chile entre enero y junio? (gráfico de
líneas).

In [ ]:
%%time
datosChile = seisMeses.loc[seisMeses['Country_Region'] == 'Chile'].copy()
datosChile['Last_Update'] = pd.to_datetime(datosChile['Last_Update'])
datosAgrupados = datosChile.groupby(datosChile['Last_Update'])['Confirmed'].sum().reset_index()
datosAgrupados

plt.figure(figsize=(12, 7))
plt.plot(datosAgrupados['Last_Update'], datosAgrupados['Confirmed'], label='Casos Confirmados', color='b')

plt.title('Evolución de Casos Confirmados en Chile (Enero - Junio 2021)', fontsize=16)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Total de Casos Confirmados', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()


6. ¿Cuál fue la fecha con más nuevos casos a nivel mundial durante este período?

In [ ]:
%%time
casosNuevos = seisMeses[['Country_Region','Last_Update','Confirmed']].copy()
casosNuevos['Last_Update'] = pd.to_datetime(casosNuevos['Last_Update'], format='ISO8601')

casosNuevos['Last_Update'] = casosNuevos['Last_Update'].dt.date

casosNuevos_mundial = casosNuevos.groupby('Last_Update')['Confirmed'].sum().reset_index()
casosNuevos_mundial['Nuevos Casos'] = casosNuevos_mundial['Confirmed'].diff()

fila_max_nuevos = casosNuevos_mundial.loc[casosNuevos_mundial['Nuevos Casos'].idxmax()]

fecha_max = fila_max_nuevos['Last_Update']
casos_max = fila_max_nuevos['Nuevos Casos']

print(f"\nLa fecha con más nuevos casos reportados a nivel mundial fue: {fecha_max}, con un total de {int(casos_max):,} casos nuevos")

7. ¿Existe correlación entre casos confirmados y fallecidos? (gráfico de dispersión +
regresión).

In [ ]:
%%time
cols_to_plot = ['Confirmed', 'Deaths']
seisMeses_paises = seisMeses.copy()

In [ ]:
%%time
plt.figure(figsize=(12, 7))

# Usamos sns.scatterplot (en lugar de regplot)
# 'alpha=0.1' hace los puntos 90% transparentes para ver la densidad
# 's=10' hace los puntos más pequeños
sb.scatterplot(
    data=seisMeses_paises, 
    x='Confirmed', 
    y='Deaths',
    alpha=0.1, 
    s=10,
    edgecolor='none' # Quitar bordes de los puntos
)

# ¡¡Crucial!! Usar escala logarítmica para que los datos se separen
plt.xscale('log')
plt.yscale('log')

plt.title('Dispersión de Casos Confirmados vs. Muertes (Registros Diarios)', fontsize=16)
plt.xlabel('Casos Confirmados (Escala Logarítmica)', fontsize=12)
plt.ylabel('Muertes (Escala Logarítmica)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5, which='both') 
plt.tight_layout()

In [ ]:
%%time
plt.figure(figsize=(12, 7))

# sns.regplot() dibuja los puntos Y la línea de regresión
sb.regplot(
    data=seisMeses_paises, 
    x='Confirmed', 
    y='Deaths',
    scatter_kws={'alpha': 0.5, 's': 50},       # Estilo de los puntos
    line_kws={'color': 'red', 'linestyle': '--'} # Estilo de la línea
)

# ¡¡Crucial!! Usar escala logarítmica en ambos ejes
plt.xscale('log')
plt.yscale('log')

plt.title('Regresión de Muertes vs. Casos Confirmados (Totales por País)', fontsize=16)
plt.xlabel('Total Casos Confirmados (Escala Logarítmica)', fontsize=12)
plt.ylabel('Total Muertes (Escala Logarítmica)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5, which='both')
plt.tight_layout()

# Guardar el Gráfico
output_filename = 'grafico_regresion_por_pais.png'
plt.savefig(output_filename)
print(f"Gráfico guardado exitosamente como: {output_filename}")

8. Mostrar el Top 10 de países con mayor crecimiento porcentual de casos entre mayo y
junio.

In [ ]:
%%time
mayorCrecimiento = seisMeses.copy()
mayorCrecimiento['Last_Update'] = pd.to_datetime(mayorCrecimiento['Last_Update'], format='ISO8601')
crecimientoDaily = mayorCrecimiento.groupby(['Country_Region', 'Last_Update'])['Confirmed'].sum().reset_index()

fecha_mayo = pd.to_datetime('2021-05-31')
fecha_junio = pd.to_datetime('2021-06-30')

#Crecimiento historico finales de mayo.
df_mayo_historico = crecimientoDaily[crecimientoDaily['Last_Update'] <= fecha_mayo]
df_mayo_final = df_mayo_historico.groupby('Country_Region')['Confirmed'].max().reset_index()
df_mayo_final.columns = ['Country_Region', 'Total_Mayo']

#crecimiento historico finales de junio
df_junio_historico = crecimientoDaily[crecimientoDaily['Last_Update'] <= fecha_junio]
df_junio_final = df_junio_historico.groupby('Country_Region')['Confirmed'].max().reset_index()
df_junio_final.columns = ['Country_Region', 'Total_Junio']

crecimientoTotal = pd.merge(df_mayo_final, df_junio_final, on='Country_Region', how='inner')
threshold = 100
df_growth_filtrado = crecimientoTotal[crecimientoTotal['Total_Mayo'] > threshold]

df_growth_filtrado['Crecimiento_Porc'] = ((df_growth_filtrado['Total_Junio'] - df_growth_filtrado['Total_Mayo']) / df_growth_filtrado['Total_Mayo']) * 100
df_top10 = df_growth_filtrado.sort_values(by='Crecimiento_Porc', ascending=False).head(10)
df_top10


9. Identificar países con rebrote (un día sin casos y luego un incremento posterior).

In [ ]:
%%time
rebrote = seisMeses.copy()
rebrote['Last_Update'] = pd.to_datetime(rebrote['Last_Update'], format='ISO8601')

rebrote = rebrote.sort_values(by=['Country_Region', 'Last_Update'])
rebrote['Nuevos_Casos'] = rebrote.groupby('Country_Region')['Confirmed'].diff()

rebrote['Nuevos_Casos'] = rebrote['Nuevos_Casos'].fillna(rebrote['Confirmed'])
rebrote['Nuevos_Casos'] = rebrote['Nuevos_Casos'].clip(lower=0)

In [ ]:
%%time
paises_con_rebrote = []
# Asumiendo que 'rebrote' ya está cargado y procesado de la celda anterior (09439694)
datos_agrupados = rebrote.groupby('Country_Region')

for nombre_pais, datos_pais in datos_agrupados:
    
    # Obtener todas las fechas donde hubo 0 casos nuevos
    fechas_cero_casos = datos_pais[datos_pais['Nuevos_Casos'] == 0]['Last_Update']
    
    # Si nunca tuvo un día con 0 casos, no puede tener rebrote
    if fechas_cero_casos.empty:
        continue
        
    # Encontrar la PRIMERA fecha con 0 casos
    primera_fecha_cero = fechas_cero_casos.min()
    
    # --- CÓDIGO CORREGIDO ---
    # Filtramos los datos para ver si hay algún día POSTERIOR con casos
    df_filtrado = datos_pais[
        (datos_pais['Last_Update'] > primera_fecha_cero) & 
        (datos_pais['Nuevos_Casos'] > 0)
    ]
    
    # Comprobamos si el DataFrame filtrado NO está vacío
    # Esto es más claro que .any().any() y elimina la advertencia.
    tiene_crecimiento_posterior = not df_filtrado.empty
    # --- FIN DE LA CORRECCIÓN ---
    
    if tiene_crecimiento_posterior:
        paises_con_rebrote.append(nombre_pais)

# --- 5. Mostrar Resultados ---
print("\n--- Países Identificados con Rebrote ---")
if paises_con_rebrote:
    for pais in paises_con_rebrote:
        print(f"- {pais}")
else:
    print("No se encontraron países con el patrón de rebrote especificado.")

10. Generar un reporte de perfilado automático (ydata-profiling o pandas_profiling) que incluya
distribuciones, correlaciones y resumen de calidad de datos.

In [ ]:
%%time
from ydata_profiling import ProfileReport


# Crea el objeto de reporte
profile = ProfileReport(seisMeses, title="Reporte de Perfilado - Datos COVID")


# --- 3. Guardar el Reporte como HTML ---
output_file = "perfilado.html"

if os.path.exists(output_file):
    os.remove(output_file)

profile.to_file(output_file)

print("\n--- ¡Reporte Generado! ---")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 14/14 [00:08<00:00,  1.69it/s]
/home/codespace/.local/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]


--- ¡Reporte Generado! ---
CPU times: user 27.2 s, sys: 1.54 s, total: 28.7 s
Wall time: 27.7 s
